In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from datetime import datetime

from utils.plots import LivePlotCallback
from utils.preprocess_data import get_garbage_datasets
from utils.model_architectures import build_experimental_model
from config import IMG_SIZE, BATCH_SIZE, CLASSES, SEED, EPOCHS
import certifi

In [2]:
# Tell SSL to use the certifi certificate bundle
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [3]:
image_categories_path = '../data/images'

In [4]:
train_ds, val_ds, class_names = get_garbage_datasets(
    data_dir=image_categories_path, 
    batch_size=BATCH_SIZE, 
    img_size=IMG_SIZE,
    model_type='custom' 
)

Found 13901 files belonging to 6 classes.
Using 11121 files for training.


2026-05-30 14:04:33.678360: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-05-30 14:04:33.678392: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-05-30 14:04:33.678398: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-05-30 14:04:33.678415: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-30 14:04:33.678426: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 13901 files belonging to 6 classes.
Using 2780 files for validation.


### Phase 1

In [ ]:
architectures_to_test = [
    'mobilenet_v2' # Phase 1 Feature Extraction
]

model_dir = '../models/experiments/'
log_dir = '../logs/experiments/'
os.makedirs(model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

experiment_histories = {}

for arch in architectures_to_test:
    print(f"\n{'='*50}")
    print(f"🚀 STARTING EXPERIMENT: {arch.upper()}")
    print(f"{'='*50}\n")
    
    # model = build_experimental_model(arch)
    model = build_experimental_model(arch)
    
    model_filepath = os.path.join(model_dir, f'{arch}_best.weights.h5') 
    csv_log_file = os.path.join(log_dir, f'{arch}_history.csv')
    
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=model_filepath, 
        monitor='val_accuracy', 
        save_best_only=True, 
        save_weights_only=True, 
        verbose=1
    )
    
    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=3, min_lr=0.00001, verbose=1
    )
    
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1
    )
    
    csv_logger = tf.keras.callbacks.CSVLogger(csv_log_file, append=False)
    
    EPOCHS = 50

    history = model.fit(
        train_ds, 
        validation_data=val_ds, 
        epochs=EPOCHS, 
        callbacks=[checkpoint, lr_scheduler, early_stop, csv_logger],
        verbose=1 
    )
    
    experiment_histories[arch] = history.history


🚀 STARTING EXPERIMENT: MOBILENET_V2



/Users/kvbek/Documents/projects/garbage-sorting-model/src/utils/model_architectures.py:101: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Epoch 1/50


2026-05-30 13:31:19.176163: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - accuracy: 0.7227 - loss: 0.8866
Epoch 1: val_accuracy improved from None to 0.86655, saving model to ../models/experiments/mobilenet_v2_best.weights.h5

Epoch 1: finished saving model to ../models/experiments/mobilenet_v2_best.weights.h5
174/174 ━━━━━━━━━━━━━━━━━━━━ 41s 215ms/step - accuracy: 0.7985 - loss: 0.6222 - val_accuracy: 0.8665 - val_loss: 0.4153 - learning_rate: 0.0010
Epoch 2/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.8715 - loss: 0.3756
Epoch 2: val_accuracy improved from 0.86655 to 0.86835, saving model to ../models/experiments/mobilenet_v2_best.weights.h5

Epoch 2: finished saving model to ../models/experiments/mobilenet_v2_best.weights.h5
174/174 ━━━━━━━━━━━━━━━━━━━━ 36s 207ms/step - accuracy: 0.8731 - loss: 0.3665 - val_accuracy: 0.8683 - val_loss: 0.3957 - learning_rate: 0.0010
Epoch 3/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.8990 - loss: 0.2970
Epoch 3: val_accuracy improved from 0.86835 

### Phase 2

In [6]:
architectures_to_test = [
    'mobilenet_v2_finetuned' # Phase 1 Feature Extraction
]

model_dir = '../models/experiments/'
log_dir = '../logs/experiments/'
os.makedirs(model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

experiment_histories = {}

for arch in architectures_to_test:
    print(f"\n{'='*50}")
    print(f"🚀 STARTING EXPERIMENT: {arch.upper()}")
    print(f"{'='*50}\n")
    
    # model = build_experimental_model(arch)
    model = build_experimental_model(arch)
    
    model_filepath = os.path.join(model_dir, f'{arch}_best.weights.h5') 
    csv_log_file = os.path.join(log_dir, f'{arch}_history.csv')
    
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=model_filepath, 
        monitor='val_accuracy', 
        save_best_only=True, 
        save_weights_only=True, 
        verbose=1
    )
    
    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=3, min_lr=0.00001, verbose=1
    )
    
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1
    )
    
    csv_logger = tf.keras.callbacks.CSVLogger(csv_log_file, append=False)
    
    EPOCHS = 50

    history = model.fit(
        train_ds, 
        validation_data=val_ds, 
        epochs=EPOCHS, 
        callbacks=[checkpoint, lr_scheduler, early_stop, csv_logger],
        verbose=1 
    )
    
    experiment_histories[arch] = history.history


🚀 STARTING EXPERIMENT: MOBILENET_V2_FINETUNED


📥 Loading Phase 1 weights from: ../models/experiment2/mobilenet_v2_best.weights.h5
🔓 Unfreezing deep MobileNetV2 layers...
Epoch 1/50


2026-05-30 14:05:29.195245: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.8253 - loss: 0.5468
Epoch 1: val_accuracy improved from None to 0.90036, saving model to ../models/experiments/mobilenet_v2_finetuned_best.weights.h5

Epoch 1: finished saving model to ../models/experiments/mobilenet_v2_finetuned_best.weights.h5
174/174 ━━━━━━━━━━━━━━━━━━━━ 68s 326ms/step - accuracy: 0.8442 - loss: 0.4837 - val_accuracy: 0.9004 - val_loss: 0.3374 - learning_rate: 1.0000e-05
Epoch 2/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - accuracy: 0.8948 - loss: 0.3193
Epoch 2: val_accuracy did not improve from 0.90036
174/174 ━━━━━━━━━━━━━━━━━━━━ 65s 355ms/step - accuracy: 0.8898 - loss: 0.3251 - val_accuracy: 0.8993 - val_loss: 0.3252 - learning_rate: 1.0000e-05
Epoch 3/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step - accuracy: 0.9072 - loss: 0.2590
Epoch 3: val_accuracy improved from 0.90036 to 0.90899, saving model to ../models/experiments/mobilenet_v2_finetuned_best.weights.h5

Epoch 3: finished saving model to ../mod